# 03 — Gold Layer: Dimensions

**Purpose:** Build all conformed dimension tables in the Gold layer from Silver staging tables.

**What this notebook does:**
- Builds all seven conformed dimensions from Silver
- Generates integer surrogate keys deterministically using `row_number()`
- Seeds `dim_date` TBD row (`date_key = 0`) — required before any fact notebook runs
- MERGEs dimensions — new values inserted, existing values updated (Type 1) or versioned (Type 2)

**Output tables:**
- `gold.dim_date` — full calendar + academic calendar (pre-populated)
- `gold.dim_student` — SCD Type 2 (all versions from silver.student)
- `gold.dim_course` — SCD Type 1
- `gold.dim_campus` — SCD Type 1
- `gold.dim_intake` — SCD Type 1
- `gold.dim_lead_source` — SCD Type 1
- `gold.dim_status` — SCD Type 1

> **Critical:** This notebook must complete before any fact notebook runs.
> Fact tables join to dimension surrogate keys — if dims don't exist, facts will fail.
> The TBD row in `dim_date` (date_key = 0) must exist before `04_gold_fact_pipeline`.

## Parameters

In [ ]:
pipeline_run_date = "2024-01-15"  # YYYY-MM-DD

## Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DateType, BooleanType
from delta.tables import DeltaTable

run_date = pipeline_run_date
print(f"Pipeline run date: {run_date}")


def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name)
        return True
    except Exception:
        return False


def add_surrogate_key(df, key_col: str, order_col: str):
    """
    Adds a deterministic integer surrogate key using row_number().
    Always ordered by natural key — same input always produces same SK.
    Never use monotonically_increasing_id() — it is non-deterministic across runs.
    """
    w = Window.orderBy(order_col)
    return df.withColumn(key_col, F.row_number().over(w).cast(IntegerType()))


print("Setup complete.")

## dim_date

Pre-populated date dimension covering all dates relevant to the demo dataset.
Includes standard calendar columns plus JMC academic calendar columns.

**Special rows:**
- `date_key = 0` — TBD (unresolved milestone dates in accumulating snapshot)

This table is built once and never updated by the pipeline.
Academic calendar columns (intake, teaching week) are populated from known intake dates.

In [ ]:
DIM_DATE = "gold.dim_date"

if table_exists(DIM_DATE):
    print(f"[dim_date] Already exists — skipping rebuild")
else:
    print("[dim_date] Building date dimension...")

    # Generate date range: 2023-01-01 to 2025-12-31
    date_range = spark.sql("""
        SELECT explode(sequence(
            to_date('2023-01-01'),
            to_date('2025-12-31'),
            interval 1 day
        )) AS full_date
    """)

    # Intake 2024-S1: Feb 26 – Jun 30, 2024 (census: Mar 31, orientation: Feb 19-23)
    intake_start  = F.lit("2024-02-26").cast(DateType())
    intake_end    = F.lit("2024-06-30").cast(DateType())
    census_date   = F.lit("2024-03-31").cast(DateType())
    orient_start  = F.lit("2024-02-19").cast(DateType())
    orient_end    = F.lit("2024-02-23").cast(DateType())

    dim_date = (
        date_range
        .withColumn("date_key",
            F.date_format(F.col("full_date"), "yyyyMMdd").cast(IntegerType()))
        .withColumn("day_of_week",
            F.date_format(F.col("full_date"), "EEEE"))
        .withColumn("day_number_in_month",
            F.dayofmonth(F.col("full_date")))
        .withColumn("month_number",
            F.month(F.col("full_date")))
        .withColumn("month_name",
            F.date_format(F.col("full_date"), "MMMM"))
        .withColumn("calendar_quarter",
            F.quarter(F.col("full_date")))
        .withColumn("calendar_year",
            F.year(F.col("full_date")))
        # Academic calendar — 2024-S1 only for demo
        .withColumn("intake_code",
            F.when(
                (F.col("full_date") >= intake_start) &
                (F.col("full_date") <= intake_end),
                F.lit("2024-S1")
            ).otherwise(F.lit(None)))
        .withColumn("intake_name",
            F.when(F.col("intake_code") == "2024-S1",
                F.lit("Semester 1 2024")
            ).otherwise(F.lit(None)))
        .withColumn("academic_year",
            F.when(F.col("intake_code").isNotNull(),
                F.lit("2024")
            ).otherwise(F.lit(None)))
        .withColumn("is_teaching_week",
            F.when(
                (F.col("full_date") >= intake_start) &
                (F.col("full_date") <= intake_end) &
                (F.col("full_date") < orient_start |
                 F.col("full_date") > orient_end),
                F.lit(True)
            ).otherwise(F.lit(False)))
        .withColumn("teaching_week_number",
            F.when(
                F.col("is_teaching_week") == True,
                F.ceil(
                    F.datediff(F.col("full_date"), intake_start) / 7
                ).cast(IntegerType())
            ).otherwise(F.lit(None)))
        .withColumn("is_census_date",
            F.col("full_date") == census_date)
    )

    # Write calendar rows
    dim_date.write.format("delta").mode("overwrite").saveAsTable(DIM_DATE)

    # Seed TBD row — date_key = 0
    # This must exist before any fact notebook runs
    # Accumulating snapshot uses date_key = 0 for unresolved milestones
    tbd_row = spark.createDataFrame([(
        0,        # date_key
        None,     # full_date
        "TBD",    # day_of_week
        None,     # day_number_in_month
        None,     # month_number
        "TBD",    # month_name
        None,     # calendar_quarter
        None,     # calendar_year
        None,     # intake_code
        None,     # intake_name
        None,     # academic_year
        False,    # is_teaching_week
        None,     # teaching_week_number
        False,    # is_census_date
    )], schema=dim_date.schema)

    tbd_row.write.format("delta").mode("append").saveAsTable(DIM_DATE)

    total = spark.table(DIM_DATE).count()
    print(f"[dim_date] Built {total} rows (includes TBD row with date_key = 0)")

# Verify TBD row exists
tbd_check = spark.table(DIM_DATE).filter(F.col("date_key") == 0).count()
print(f"[dim_date] TBD row (date_key=0) exists: {tbd_check == 1}")

## dim_student

Built directly from `silver.student` — all versions, not just current.
SCD Type 2 is already handled in Silver — Gold just needs surrogate keys assigned.

**Surrogate key strategy:**
SK is assigned per (student_id, _valid_from) combination.
This ensures a new SK is generated for each Type 2 version.

In [ ]:
DIM_STUDENT = "gold.dim_student"

silver_student = spark.table("silver.student")

incoming_students = (
    silver_student
    .withColumn(
        "student_type",
        F.when(F.col("is_international") == True, F.lit("International"))
         .otherwise(F.lit("Domestic"))
    )
    .withColumn(
        "visa_description",
        F.when(F.col("visa_code") == "Domestic", F.lit("Domestic Student"))
         .otherwise(F.lit("International Student Visa"))
    )
    .select(
        "student_id",
        "student_first_name",
        "student_last_name",
        "student_email",
        "date_of_birth",
        "visa_code",
        "visa_description",
        "is_international",
        "student_type",
        "lead_source",
        "_valid_from",
        "_valid_to",
        "_is_current",
    )
)

if not table_exists(DIM_STUDENT):
    print("[dim_student] First run — creating table")
    # Assign surrogate keys ordered by student_id + _valid_from
    w = Window.orderBy("student_id", "_valid_from")
    dim_student = incoming_students.withColumn(
        "student_sk", F.row_number().over(w).cast(IntegerType())
    )
    dim_student.write.format("delta").mode("overwrite").saveAsTable(DIM_STUDENT)
else:
    print("[dim_student] Merging new versions")
    existing = spark.table(DIM_STUDENT)
    max_sk = existing.agg(F.max("student_sk")).collect()[0][0] or 0

    # Find versions in Silver not yet in Gold
    new_versions = incoming_students.join(
        existing.select("student_id", "_valid_from"),
        on=["student_id", "_valid_from"],
        how="left_anti"
    )

    new_count = new_versions.count()
    if new_count > 0:
        w = Window.orderBy("student_id", "_valid_from")
        new_versions = new_versions.withColumn(
            "student_sk",
            (F.row_number().over(w) + F.lit(max_sk)).cast(IntegerType())
        )
        new_versions.write.format("delta").mode("append").saveAsTable(DIM_STUDENT)

        # Expire old versions — sync _is_current from Silver
        DeltaTable.forName(spark, DIM_STUDENT).alias("t").merge(
            incoming_students.alias("s"),
            "t.student_id = s.student_id AND t._valid_from = s._valid_from"
        ).whenMatchedUpdate(set={
            "_valid_to":   "s._valid_to",
            "_is_current": "s._is_current"
        }).execute()

    print(f"[dim_student] New versions inserted: {new_count}")

total = spark.table(DIM_STUDENT).count()
current = spark.table(DIM_STUDENT).filter("_is_current = true").count()
print(f"[dim_student] Total rows: {total} | Current: {current}")
spark.table(DIM_STUDENT).select(
    "student_sk", "student_id", "visa_code", "student_type", "_valid_from", "_valid_to", "_is_current"
).orderBy("student_id", "_valid_from").show(20, truncate=False)

## dim_course

In [ ]:
DIM_COURSE = "gold.dim_course"

# Enrich course reference with known attributes
# In production these would come from a course master in Paradigm
course_attrs = spark.createDataFrame([
    ("BMUS301", "Bachelor of Music",                     "Bachelor",  7, 120, 6, True,  True),
    ("BDES401", "Bachelor of Design",                    "Bachelor",  7, 120, 6, True,  True),
    ("MENT501", "Master of Entertainment Business",      "Master",    9, 160, 4, True,  True),
], ["course_id", "course_name", "qualification_level",
    "aqf_level", "credit_points", "duration_semesters",
    "is_cricos_registered", "is_active"])

silver_courses = spark.table("silver.course")

incoming_courses = (
    silver_courses
    .join(course_attrs, silver_courses.course_code == course_attrs.course_id, how="left")
    .select(
        silver_courses.course_code.alias("course_id"),
        F.coalesce(course_attrs.course_name, silver_courses.course_code).alias("course_name"),
        F.coalesce(course_attrs.qualification_level, F.lit("Unknown")).alias("qualification_level"),
        F.coalesce(course_attrs.aqf_level, F.lit(0)).alias("aqf_level"),
        F.coalesce(course_attrs.credit_points, F.lit(0)).alias("credit_points"),
        F.coalesce(course_attrs.duration_semesters, F.lit(0)).alias("duration_semesters"),
        F.coalesce(course_attrs.is_cricos_registered, F.lit(False)).alias("is_cricos_registered"),
        F.coalesce(course_attrs.is_active, F.lit(True)).alias("is_active"),
    )
)

if not table_exists(DIM_COURSE):
    dim_course = add_surrogate_key(incoming_courses, "course_sk", "course_id")
    dim_course.write.format("delta").mode("overwrite").saveAsTable(DIM_COURSE)
else:
    existing = spark.table(DIM_COURSE)
    max_sk = existing.agg(F.max("course_sk")).collect()[0][0] or 0
    new_courses = incoming_courses.join(
        existing.select("course_id"), on="course_id", how="left_anti"
    )
    if new_courses.count() > 0:
        new_courses = add_surrogate_key(new_courses, "course_sk", "course_id")
        new_courses = new_courses.withColumn(
            "course_sk", (F.col("course_sk") + F.lit(max_sk)).cast(IntegerType())
        )
        new_courses.write.format("delta").mode("append").saveAsTable(DIM_COURSE)

print(f"[dim_course] Total rows: {spark.table(DIM_COURSE).count()}")
spark.table(DIM_COURSE).show(truncate=False)

## dim_campus

In [ ]:
DIM_CAMPUS = "gold.dim_campus"

campus_attrs = spark.createDataFrame([
    ("SYD", "Sydney Campus",    "Sydney",    "NSW", "Australia", "On Campus", True),
    ("MEL", "Melbourne Campus", "Melbourne", "VIC", "Australia", "On Campus", True),
], ["campus_id", "campus_name", "city", "state_code",
    "country", "delivery_mode", "is_active"])

silver_campus = spark.table("silver.campus")

incoming_campus = (
    silver_campus
    .join(campus_attrs, silver_campus.campus_code == campus_attrs.campus_id, how="left")
    .select(
        silver_campus.campus_code.alias("campus_id"),
        F.coalesce(campus_attrs.campus_name, silver_campus.campus_code).alias("campus_name"),
        F.coalesce(campus_attrs.city, F.lit("Unknown")).alias("city"),
        F.coalesce(campus_attrs.state_code, F.lit("Unknown")).alias("state_code"),
        F.coalesce(campus_attrs.country, F.lit("Australia")).alias("country"),
        F.coalesce(campus_attrs.delivery_mode, F.lit("On Campus")).alias("delivery_mode"),
        F.coalesce(campus_attrs.is_active, F.lit(True)).alias("is_active"),
    )
)

if not table_exists(DIM_CAMPUS):
    dim_campus = add_surrogate_key(incoming_campus, "campus_sk", "campus_id")
    dim_campus.write.format("delta").mode("overwrite").saveAsTable(DIM_CAMPUS)
else:
    existing = spark.table(DIM_CAMPUS)
    max_sk = existing.agg(F.max("campus_sk")).collect()[0][0] or 0
    new_campus = incoming_campus.join(
        existing.select("campus_id"), on="campus_id", how="left_anti"
    )
    if new_campus.count() > 0:
        new_campus = add_surrogate_key(new_campus, "campus_sk", "campus_id")
        new_campus = new_campus.withColumn(
            "campus_sk", (F.col("campus_sk") + F.lit(max_sk)).cast(IntegerType())
        )
        new_campus.write.format("delta").mode("append").saveAsTable(DIM_CAMPUS)

print(f"[dim_campus] Total rows: {spark.table(DIM_CAMPUS).count()}")
spark.table(DIM_CAMPUS).show(truncate=False)

## dim_intake

In [ ]:
DIM_INTAKE = "gold.dim_intake"

intake_attrs = spark.createDataFrame([
    ("2024-S1", "Semester 1 2024", "2024", 1,
     20240226, 20240630, 20240331),
], ["intake_id", "intake_name", "academic_year", "semester_number",
    "start_date_key", "end_date_key", "census_date_key"])

silver_intake = spark.table("silver.intake")

incoming_intake = (
    silver_intake
    .join(intake_attrs, silver_intake.intake_code == intake_attrs.intake_id, how="left")
    .select(
        silver_intake.intake_code.alias("intake_id"),
        F.coalesce(intake_attrs.intake_name, silver_intake.intake_code).alias("intake_name"),
        F.coalesce(intake_attrs.academic_year, F.lit("Unknown")).alias("academic_year"),
        F.coalesce(intake_attrs.semester_number, F.lit(0)).alias("semester_number"),
        F.coalesce(intake_attrs.start_date_key, F.lit(0)).alias("start_date_key"),
        F.coalesce(intake_attrs.end_date_key, F.lit(0)).alias("end_date_key"),
        F.coalesce(intake_attrs.census_date_key, F.lit(0)).alias("census_date_key"),
    )
)

if not table_exists(DIM_INTAKE):
    dim_intake = add_surrogate_key(incoming_intake, "intake_sk", "intake_id")
    dim_intake.write.format("delta").mode("overwrite").saveAsTable(DIM_INTAKE)
else:
    existing = spark.table(DIM_INTAKE)
    max_sk = existing.agg(F.max("intake_sk")).collect()[0][0] or 0
    new_intake = incoming_intake.join(
        existing.select("intake_id"), on="intake_id", how="left_anti"
    )
    if new_intake.count() > 0:
        new_intake = add_surrogate_key(new_intake, "intake_sk", "intake_id")
        new_intake = new_intake.withColumn(
            "intake_sk", (F.col("intake_sk") + F.lit(max_sk)).cast(IntegerType())
        )
        new_intake.write.format("delta").mode("append").saveAsTable(DIM_INTAKE)

print(f"[dim_intake] Total rows: {spark.table(DIM_INTAKE).count()}")
spark.table(DIM_INTAKE).show(truncate=False)

## dim_lead_source

In [ ]:
DIM_LEAD_SOURCE = "gold.dim_lead_source"

source_attrs = spark.createDataFrame([
    ("Agent",        "Referral"),
    ("Direct",       "Organic"),
    ("Social Media", "Paid"),
    ("Web Referral", "Organic"),
    ("Unknown",      "Unknown"),
], ["source_name", "source_category"])

if not table_exists(DIM_LEAD_SOURCE):
    dim_lead_source = add_surrogate_key(
        source_attrs.withColumnRenamed("source_name", "lead_source_id"),
        "lead_source_sk", "lead_source_id"
    ).withColumnRenamed("lead_source_id", "source_name") \
     .withColumn("lead_source_id", F.col("source_name"))

    # Rebuild cleanly
    w = Window.orderBy("source_name")
    dim_lead_source = (
        source_attrs
        .withColumn("lead_source_sk", F.row_number().over(w).cast(IntegerType()))
        .withColumn("lead_source_id", F.col("source_name"))
        .select("lead_source_sk", "lead_source_id", "source_name", "source_category")
    )
    dim_lead_source.write.format("delta").mode("overwrite").saveAsTable(DIM_LEAD_SOURCE)

print(f"[dim_lead_source] Total rows: {spark.table(DIM_LEAD_SOURCE).count()}")
spark.table(DIM_LEAD_SOURCE).show(truncate=False)

## dim_status

Reusable across all fact tables.
`status_category` distinguishes which process the status belongs to.

In [ ]:
DIM_STATUS = "gold.dim_status"

status_data = spark.createDataFrame([
    # Enrolment statuses
    ("Active",     "Currently enrolled",         "Enrolment",   False),
    ("Withdrawn",  "Withdrawn by student",        "Enrolment",   True),
    ("Completed",  "Successfully completed",      "Enrolment",   True),
    ("Deferred",   "Deferred to future semester", "Enrolment",   False),
    ("Excluded",   "Excluded by institution",     "Enrolment",   True),
    # Application statuses
    ("Offered",    "Offer made",                  "Application", False),
    ("Accepted",   "Offer accepted",              "Application", True),
    ("Rejected",   "Application rejected",        "Application", True),
    ("Pending",    "Awaiting decision",           "Application", False),
    # Unknown
    ("Unknown",    "Unknown status",              "Unknown",     False),
], ["status_code", "status_description", "status_category", "is_terminal"])

if not table_exists(DIM_STATUS):
    w = Window.orderBy("status_category", "status_code")
    dim_status = (
        status_data
        .withColumn("status_sk", F.row_number().over(w).cast(IntegerType()))
        .withColumn("status_id",
            F.concat(F.col("status_code"), F.lit("_"), F.col("status_category")))
        .select("status_sk", "status_id", "status_code",
                "status_description", "status_category", "is_terminal")
    )
    dim_status.write.format("delta").mode("overwrite").saveAsTable(DIM_STATUS)

print(f"[dim_status] Total rows: {spark.table(DIM_STATUS).count()}")
spark.table(DIM_STATUS).show(truncate=False)

## Final Verification

In [ ]:
gold_dims = [
    "gold.dim_date",
    "gold.dim_student",
    "gold.dim_course",
    "gold.dim_campus",
    "gold.dim_intake",
    "gold.dim_lead_source",
    "gold.dim_status",
]

print(f"{'Table':<30} {'Rows':>8}")
print("-" * 40)
for table in gold_dims:
    print(f"{table:<30} {spark.table(table).count():>8}")
print("-" * 40)

# Critical: TBD row must exist
tbd = spark.table("gold.dim_date").filter("date_key = 0").count()
print(f"\nTBD row in dim_date (date_key=0): {'PASS' if tbd == 1 else 'FAIL — fact notebooks will break'}")

print("\n[dim_student] All versions:")
spark.table("gold.dim_student").select(
    "student_sk", "student_id", "visa_code", "_valid_from", "_valid_to", "_is_current"
).orderBy("student_id", "_valid_from").show(20, truncate=False)

## Summary

All Gold dimensions built and verified.
TBD row seeded in `dim_date` — fact notebooks can now run safely.

**Next steps:** Run notebooks 04, 05, 06 in any order.